# GENESIS — Benchmark Results

Latency and quality benchmarks across Gemma 4 vs Gemma 3 variants.

In [ ]:
import sys; sys.path.insert(0, '..')
import os, time, json
from dotenv import load_dotenv; load_dotenv('../.env')
HF_TOKEN = os.getenv('HF_TOKEN', '')
from core.gemma_engine import GemmaEngine, GEMMA_MODELS
print('Models to benchmark:', list(GEMMA_MODELS.keys()))

In [ ]:
BENCH_PROMPTS = [
    ('factual',   'What year was the Eiffel Tower built?', 20),
    ('reasoning', 'If A>B and B>C, is A>C? Explain.', 80),
    ('creative',  'Write a 2-sentence haiku about neural networks.', 60),
    ('code',      'Write a Python function to compute fibonacci(n).', 150),
    ('json',      'Return JSON: {"capital": "France", "city": str}', 50),
]

MODELS_TO_TEST = ['default', 'fast', 'mini', 'g3', 'g3-fast']

results = {}
for model_key in MODELS_TO_TEST:
    engine = GemmaEngine(token=HF_TOKEN, model=model_key)
    results[model_key] = []
    for task, prompt, max_tok in BENCH_PROMPTS:
        t0 = time.time()
        resp = engine.think(prompt, max_tokens=max_tok)
        elapsed = round(time.time() - t0, 2)
        results[model_key].append({'task': task, 'latency_s': elapsed, 'chars': len(resp)})
        print(f'{model_key:12} {task:10} {elapsed:.1f}s  {len(resp)} chars')

print('\nBenchmark complete.')

In [ ]:
import pandas as pd
rows = []
for model_key, tasks in results.items():
    for t in tasks:
        rows.append({'model': model_key, **t})
df = pd.DataFrame(rows)
print(df.groupby('model')['latency_s'].agg(['mean','min','max']).round(2))